# Appendix 13. MLflow 기초

## 1. Goal

이 notebook은 MLflow Tracking의 Experiment, Run, parameter, metric, tag, dataset, artifact와 logged model 관계를 익히는 자료입니다. 작은 scikit-learn model을 학습해 수동으로 기록하고, API로 다시 조회하고 load합니다.

MLflow는 model 자체의 품질을 보장하지 않습니다. 어떤 code와 data, 설정으로 어떤 결과와 artifact가 만들어졌는지 추적할 수 있게 하는 것이 이 실습의 핵심입니다.

## 2. Setup

프로젝트에 설치된 MLflow 3.x를 사용합니다. 외부 tracking server 대신 임시 디렉터리의 SQLite backend와 local artifact storage를 사용하고 마지막 Checks에서 정리합니다. 기존 `mlruns`, model registry와 repository artifact는 변경하지 않습니다.

In [ ]:
import tempfile
from pathlib import Path

import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

print({"mlflow": mlflow.__version__})


## 3. Steps

### 3-1. Tracking 환경과 Experiment

#### 3-1-1. local tracking URI 설정

tracking URI는 Run metadata를 어디에 기록할지 정합니다. 여기서는 SQLite file을 backend store로 사용합니다. `set_experiment()`는 같은 목적의 Run을 묶을 Experiment를 선택하며, 이름이 없으면 생성합니다. 공유 환경에서는 URI와 credential을 application setting으로 주입합니다.

In [ ]:
temporary_directory = tempfile.TemporaryDirectory(prefix="mlflow-appendix-")
tracking_root = Path(temporary_directory.name)
database_path = (tracking_root / "mlflow.db").resolve()
tracking_uri = f"sqlite:///{database_path.as_posix()}"

mlflow.set_tracking_uri(tracking_uri)
experiment = mlflow.set_experiment("appendix-local-tracking")
print(
    {
        "tracking_uri_scheme": mlflow.get_tracking_uri().split(":", 1)[0],
        "experiment_id": experiment.experiment_id,
        "experiment_name": experiment.name,
    }
)


#### 3-1-2. Experiment, Run과 logged model

Experiment는 같은 분석 목적의 Run을 묶고, Run은 한 번의 실행에서 생긴 parameter, metric, tag와 artifact를 기록합니다. MLflow 3의 logged model은 고유한 model ID와 artifact를 가지며 Run과 연결됩니다. Registry의 model name·version·alias는 검토와 배포 lifecycle을 위한 별도 개념입니다.

In [ ]:
features, target = make_classification(
    n_samples=100,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    weights=[0.65, 0.35],
    random_state=42,
)
feature_names = ["feature_1", "feature_2", "feature_3", "feature_4"]
feature_frame = pd.DataFrame(features, columns=feature_names)
target_series = pd.Series(target, name="target")
X_train, X_valid, y_train, y_valid = train_test_split(
    feature_frame,
    target_series,
    test_size=0.25,
    random_state=42,
    stratify=target_series,
)
print({"train_shape": X_train.shape, "valid_shape": X_valid.shape})


### 3-2. Run에 metadata와 artifact 기록

#### 3-2-1. parameter, metric과 tag

parameter는 Run의 입력 설정, metric은 숫자로 표현한 결과, tag는 검색과 설명에 쓰는 문자열 metadata입니다. metric에는 시간이나 step에 따른 여러 값을 기록할 수 있습니다. 여기서는 한 번의 validation 결과를 명시적으로 기록합니다.

In [ ]:
model_parameters = {"C": 1.0, "max_iter": 1000, "random_state": 42}
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(**model_parameters),
)

with mlflow.start_run(run_name="manual-logistic-regression") as active_run:
    model.fit(X_train, y_train)
    valid_probabilities = model.predict_proba(X_valid)[:, 1]
    run_metrics = {
        "valid_roc_auc": roc_auc_score(y_valid, valid_probabilities),
        "valid_average_precision": average_precision_score(
            y_valid, valid_probabilities
        ),
    }
    mlflow.log_params(model_parameters)
    mlflow.log_metrics(run_metrics)
    mlflow.set_tags(
        {
            "data_role": "synthetic-train-valid",
            "purpose": "appendix-api-walkthrough",
        }
    )
    run_id = active_run.info.run_id

print({"run_id": run_id, **run_metrics})


#### 3-2-2. dataset input과 일반 artifact

dataset input은 Run이 사용한 data의 이름, digest, schema와 source를 추적합니다. artifact는 JSON evidence, chart, model bundle처럼 숫자 metric으로 표현할 수 없는 파일입니다. 같은 Run에 추가로 기록하려면 `start_run(run_id=...)`로 기존 Run을 다시 활성화할 수 있습니다.

In [ ]:
training_dataset = mlflow.data.from_pandas(
    X_train.assign(target=y_train),
    name="synthetic-training-data",
    targets="target",
)

with mlflow.start_run(run_id=run_id):
    mlflow.log_input(training_dataset, context="training")
    mlflow.log_dict(
        {
            "dataset_role": "synthetic-training",
            "validation_row_count": len(X_valid),
            "metrics": run_metrics,
        },
        artifact_file="evidence/summary.json",
    )

print({"dataset_digest": training_dataset.digest})


#### 3-2-3. signature와 함께 model 기록

model signature는 입력과 출력 schema를 기록하고, input example은 예상 입력 형태를 보여줍니다. MLflow 3에서는 `name`으로 logged model을 만들며 반환된 `model_uri`로 다시 load할 수 있습니다. serialization만 성공했다고 model 승인이나 배포 준비가 끝난 것은 아닙니다.

In [ ]:
input_example = X_train.head(3)
signature = infer_signature(X_train, model.predict_proba(X_train))

with mlflow.start_run(run_id=run_id):
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="classifier",
        signature=signature,
        input_example=input_example,
    )

print({"model_id": model_info.model_id, "model_uri": model_info.model_uri})


### 3-3. Run 조회와 model load

#### 3-3-1. MlflowClient와 search_runs

fluent API는 현재 활성화된 Run에 기록할 때 편리하고, `MlflowClient`는 ID로 metadata와 artifact를 조회할 때 유용합니다. `search_runs()`는 여러 Run을 비교할 수 있는 table을 반환합니다. production code에서는 system tag 이름과 provenance key를 owning contract에서 관리합니다.

In [ ]:
client = MlflowClient()
logged_run = client.get_run(run_id)
artifact_paths = [item.path for item in client.list_artifacts(run_id)]
run_table = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.purpose = 'appendix-api-walkthrough'",
    order_by=["metrics.valid_roc_auc DESC"],
)
print(
    {
        "parameter_C": logged_run.data.params["C"],
        "metric_names": sorted(logged_run.data.metrics),
        "root_artifacts": artifact_paths,
        "matched_runs": len(run_table),
    }
)


#### 3-3-2. pyfunc flavor로 model load

MLflow model은 여러 flavor를 가질 수 있습니다. `mlflow.pyfunc.load_model()`은 공통 `predict()` interface를 제공하고, `mlflow.sklearn.load_model()`은 native scikit-learn 객체가 필요할 때 사용합니다. load 후에는 동일한 input example과 feature contract로 동작을 다시 확인합니다.

In [ ]:
loaded_pyfunc_model = mlflow.pyfunc.load_model(model_info.model_uri)
loaded_predictions = loaded_pyfunc_model.predict(X_valid.head(5))
native_model = mlflow.sklearn.load_model(model_info.model_uri)
native_predictions = native_model.predict(X_valid.head(5))

print(
    {
        "pyfunc_prediction_shape": loaded_predictions.shape,
        "native_prediction_shape": native_predictions.shape,
    }
)


#### 3-3-3. autologging을 사용할 때 확인할 점

`mlflow.sklearn.autolog()`는 estimator의 parameter, metric과 model을 자동 기록합니다. 빠른 실험에는 편리하지만 무엇이 기록되는지 library와 version에 따라 달라질 수 있습니다. release evidence처럼 contract가 중요한 흐름에서는 필요한 key를 수동으로 기록하고 test하는 편이 명확합니다. 이 notebook은 중복 Run 생성을 피하기 위해 autologging을 실행하지 않습니다.

In [ ]:
autolog_api = mlflow.sklearn.autolog
print({"callable": callable(autolog_api), "name": autolog_api.__name__})


## 4. Checks

Run metadata, artifact, model URI와 load 결과가 일치하는지 확인한 뒤 임시 tracking storage를 정리합니다.

In [ ]:
assert database_path.exists()
assert logged_run.data.params["C"] == "1.0"
assert set(run_metrics) <= set(logged_run.data.metrics)
assert "evidence" in artifact_paths
assert len(run_table) == 1
assert model_info.model_id
assert loaded_predictions.shape == native_predictions.shape == (5,)
assert (loaded_predictions == native_predictions).all()
temporary_directory.cleanup()
print("All appendix checks passed.")


## 5. Next Steps

- Experiment는 task 기준으로, Run은 재현 가능한 한 번의 실행 기준으로 나눕니다.
- parameter, metric, tag와 artifact의 역할을 구분하고 key를 contract로 관리합니다.
- dataset digest, Git revision, DVC lock digest와 config hash를 Run에 함께 기록합니다.
- model signature와 input example을 남기고 load 후 prediction contract를 검증합니다.
- Registry alias만으로 deployment artifact를 식별하지 않고 immutable digest와 release manifest를 함께 사용합니다.

## 6. References

이 notebook의 설명과 예제는 다음 공식 문서를 기준으로 작성했습니다.

- [MLflow Tracking Quickstart](https://mlflow.org/docs/latest/ml/getting-started/quickstart/)
- [MLflow Tracking](https://mlflow.org/docs/latest/ml/tracking/)
- [MLflow Tracking APIs](https://mlflow.org/docs/latest/ml/tracking/tracking-api/)
- [MLflow Data](https://mlflow.org/docs/latest/ml/dataset/)
- [MLflow Model Signatures](https://mlflow.org/docs/latest/ml/model/signatures/)
- [MLflow scikit-learn API](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.sklearn.html)